# Creating Volume For Storing Raw Data

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS ottconsumption.raw.raw_files
LOCATION 'abfss://ott-consumption@adlsdemodatabrickseus202.dfs.core.windows.net/raw_files'
COMMENT 'External Volume Location';

In [0]:
%sql
-- DROP VOLUME ottconsumption.raw.raw_files;

In [0]:
%sql
LIST '/Volumes/ottconsumption/raw/raw_files';
-- REMOVE '/Volumes/ottconsumption/bronze/raw_files';

In [0]:
# dbutils.fs.rm("dbfs:/Volumes/ottconsumption/raw/raw_files", True)

# STEP 1 — Create Metadata Table (Driver of Everything)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ottconsumption.bronze.metadata_config (
    table_name STRING,
    source_path STRING,
    target_table STRING,
    file_format STRING,
    watermark_column STRING,
    is_active STRING
)
USING DELTA;

## Insert Metadata Into metadata_config for 4 Tables

In [0]:
%sql
-- INSERT INTO ottconsumption.bronze.metadata_config VALUES
-- ('users', '/Volumes/ottconsumption/raw/raw_files/users/', 'ottconsumption.bronze.users', 'csv', 'signup_date', 'Y'),

-- ('subscriptions', '/Volumes/ottconsumption/raw/raw_files/subscriptions/', 'ottconsumption.bronze.subscriptions', 'csv', 'start_date', 'Y'),

-- ('payments', '/Volumes/ottconsumption/raw/raw_files/payments/', 'ottconsumption.bronze.payments', 'csv', 'payment_date', 'Y'),

-- ('watch_history', '/Volumes/ottconsumption/raw/raw_files/watch_history/', 'ottconsumption.bronze.watch_history', 'csv', 'watch_date', 'Y');

In [0]:
%sql
-- DELETE FROM ottconsumption.bronze.metadata_config;

In [0]:
%sql
SELECT * FROM ottconsumption.bronze.metadata_config;

In [0]:
dff= spark.read.format('csv').option('header', True).load("/Volumes/ottconsumption/raw/raw_files/users/")

dff.display()

In [0]:
# from pyspark.sql.functions import col,max
# dff.agg(max('signup_date')).collect()[0][0]

# STEP 2 — Create Watermark Table (Incremental Logic)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ottconsumption.bronze.watermark_table (
    table_name STRING,
    last_processed_value TIMESTAMP
)
USING DELTA;

# STEP 3 — Add Audit Table (Industry Standard)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ottconsumption.bronze.audit_log (
    table_name STRING,
    record_count INT,
    ingestion_time TIMESTAMP,
    status STRING
)
USING DELTA;

# STEP 4 — Bronze Ingestion Function (Dynamic)

In [0]:
%sql
SELECT last_processed_value 
FROM ottconsumption.bronze.watermark_table 
-- WHERE table_name = '{table_name}'

## Creating dbutils.widgets For Dynamic Metadata Config

In [0]:
dbutils.widgets.text("metadata_config", "")

## Extracting Widget Output

In [0]:
metadata_row = dbutils.widgets.get("metadata_config")

In [0]:
print(metadata_row)

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, input_file_name, col, lit, max
import json

metadata_row = json.loads(metadata_row)
table_name = metadata_row["table_name"]
source_path = metadata_row["source_path"]
target_table = metadata_row["target_table"]
watermark_col = metadata_row["watermark_column"]

## Step 1: Read Data

In [0]:
print(f"Processing table: {table_name}")

df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(source_path)

## Step 2: Get Watermark

In [0]:
wm_df = spark.sql(f"""
    SELECT last_processed_value 
    FROM ottconsumption.bronze.watermark_table 
    WHERE table_name = '{table_name}'
""")

display(wm_df)

if wm_df.count() > 0:
        last_wm = wm_df.collect()[0][0]
        df = df.filter(col(watermark_col) > last_wm)

## Step 3: Add Metadata

In [0]:
df = df.withColumn("ingestion_time", current_timestamp()) \
        .withColumn("ingestion_date", current_date()) \
        .withColumn("source_file", col("_metadata.file_path"))

## Step 4: Write to Bronze

In [0]:
df.limit(10).display()

In [0]:
df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)
    

## Step 5: Update Watermark Using MERGE INTO

In [0]:
new_wm = df.agg(max(watermark_col)).collect()[0][0]

if new_wm is not None:
    print(f"Updating watermark to {new_wm}")
    spark.sql(f"""
        MERGE INTO ottconsumption.bronze.watermark_table t
        USING (SELECT '{table_name}' AS table_name, TIMESTAMP('{new_wm}') AS last_processed_value) s
        ON t.table_name = s.table_name
        WHEN MATCHED THEN UPDATE SET t.last_processed_value = s.last_processed_value
        WHEN NOT MATCHED THEN INSERT *
    """)


## Step 6: Log Success

In [0]:
record_count = df.count()

spark.sql(f"""
INSERT INTO ottconsumption.bronze.audit_log 
VALUES ('{table_name}', '{record_count}', current_timestamp(), 'SUCCESS')
""")

print(f"{table_name} ingestion complete")

In [0]:
# from pyspark.sql.functions import current_timestamp, current_date, input_file_name, col, lit, max

# def ingest_bronze(metadata_row):
    
#     table_name = metadata_row.table_name
#     source_path = metadata_row.source_path
#     target_table = metadata_row.target_table
#     watermark_col = metadata_row.watermark_column
    
#     print(f"Processing table: {table_name}")
    
#     # Step 1: Read Data
#     df = spark.read.format("csv") \
#         .option("header", True) \
#         .option("inferSchema", True) \
#         .load(source_path)
    
#     # Step 2: Get Watermark
#     wm_df = spark.sql(f"""
#         SELECT last_processed_value 
#         FROM ottconsumption.bronze.watermark_table 
#         WHERE table_name = '{table_name}'
#     """)
    
#     if wm_df.count() > 0:
#         last_wm = wm_df.collect()[0][0]
#         df = df.filter(col(watermark_col) > last_wm)
    
#     # Step 3: Add Metadata
#     df = df.withColumn("ingestion_time", current_timestamp()) \
#            .withColumn("ingestion_date", current_date()) \
#            .withColumn("source_file", col("_metadata.file_path"))
    
#     # Step 4: Write to Bronze
#     df.write.format("delta") \
#         .mode("append") \
#         .option("mergeSchema", "true") \
#         .saveAsTable(target_table)
    
#     # Step 5: Update Watermark
#     new_wm = df.agg(max(watermark_col)).collect()[0][0]
    
#     spark.sql(f"""
#         MERGE INTO ottconsumption.bronze.watermark_table t
#         USING (SELECT '{table_name}' AS table_name, TIMESTAMP('{new_wm}') AS last_processed_value) s
#         ON t.table_name = s.table_name
#         WHEN MATCHED THEN UPDATE SET t.last_processed_value = s.last_processed_value
#         WHEN NOT MATCHED THEN INSERT *
#     """)
    
#     # Step 6: Log Success
#     record_count = df.count()

#     spark.sql(f"""
#     INSERT INTO ottconsumption.bronze.audit_log 
#     VALUES ('{table_name}', {record_count}, current_timestamp(), 'SUCCESS')
#     """)
    
#     print(f"{table_name} ingestion complete")

## To View _metadata.* Column

In [0]:
df.select("_metadata.*").show(truncate=False)

## Delete Rows From All the Tables

In [0]:
%sql
-- DELETE FROM ottconsumption.bronze.watermark_table;
-- DELETE FROM ottconsumption.bronze.metadata_config;
-- DELETE FROM ottconsumption.bronze.audit_log;
-- DELETE FROM ottconsumption.bronze.watch_history;
-- DELETE FROM ottconsumption.bronze.subscriptions;
-- DELETE FROM ottconsumption.bronze.payments;
-- DELETE FROM ottconsumption.bronze.users;

# STEP 5 — Run for All Tables (Dynamic Execution)

In [0]:
# metadata_df = spark.table("ottconsumption.bronze.metadata_config") \
#                    .filter("is_active = 'Y'")

# for row in metadata_df.collect():
#   # print(row.table_name)
#   ingest_bronze(row)

# # metadata_df.collect()

# STEP 6 — Bronze → Silver (Preview)

## Silver Metadata Table

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS ottconsumption.silver.metadata_config (
#     source_table STRING,
#     target_table STRING,
#     watermark_column STRING
# )
# USING DELTA;

## Silver Incremental Read

In [0]:
# df = spark.table("ottconsumption.bronze.users")

# wm = spark.sql("""
# SELECT last_processed_value FROM ottconsumption.silver.watermark_table 
# WHERE table_name = 'users'
# """)

# df = df.filter(col("ingestion_time") > wm)

## Silver Merge (UPSERT)

In [0]:
# %sql

# MERGE INTO ottconsumption.silver.users t
# USING updates s
# ON t.user_id = s.user_id
# WHEN MATCHED THEN UPDATE SET *
# WHEN NOT MATCHED THEN INSERT *

# 🚀 Why This Is Industry Level

**You are now using:**

- Metadata-driven pipelines
- Dynamic ingestion
- Watermark incremental logic
- Merge-based state tracking
- Scalable multi-table ingestion

**This is exactly what:**

- Databricks projects
- Azure Data Factory pipelines
- Production ETL systems use

# Improvements

Create Logger Table more Production-Ready Audit Design
```
table_name        STRING
record_count      BIGINT
ingestion_time    TIMESTAMP
status            STRING
run_id            STRING
pipeline_name     STRING
error_message     STRING
duration_seconds  DOUBLE
```

# 🧠 Interview Explanation (🔥 MUST REMEMBER)

**If interviewer asks:**

👉 “How did you design ingestion?”

You say:
> I implemented a metadata-driven ingestion framework where ingestion is dynamically controlled using a metadata table. I used watermark-based incremental loading to process only new data and stored the progress in a watermark table. The pipeline supports multiple tables and schema evolution and writes data into Bronze Delta tables.